# DataHek OSS — 01 · Quick start

End-to-end in one cell: create the demo database, wire an offline stub
model, and ask a question through the *real* API pipeline
(schema discovery → plan → validate → guardrails → execute → explain).

Everything here runs without credentials or network access.

In [1]:
import sys, os
for _c in (os.getcwd(), os.path.join(os.getcwd(), "notebooks"), os.path.join(os.path.dirname(os.getcwd()), "notebooks")):
    if os.path.exists(os.path.join(_c, "datahek_demo.py")):
        sys.path.insert(0, _c)
        break
import tempfile
os.environ.setdefault("DATAHEK_DB_PATH", os.path.join(tempfile.gettempdir(), "datahek_notebooks.db"))

from datahek_demo import build_demo_db, make_client, show_rows

db = build_demo_db()
print("demo database:", db)

demo database: C:\Users\azcom\AppData\Local\Temp\datahek_demo.db


In [2]:
client, container = make_client(db)

print("health:", client.get("/health").json()["status"])
print("connections:", [c["name"] for c in client.get("/connections").json()])

health: ok
connections: ['demo']


## Ask a question

In [3]:
r = client.post("/ask", json={"question": "How many traces are there?", "connection_id": "conn_demo"})
print("answer:", r.json()["answer"])
show_rows(r.json()["rows"])

answer: Stub explanation: the result is shown in the table below.
total_traces=8


## Ask for an aggregate

In [4]:
r = client.post("/ask", json={"question": "What is the average duration per service?", "connection_id": "conn_demo"})
show_rows(r.json()["rows"])

service=auth-service | avg_duration=675.0
service=order-service | avg_duration=487.5
service=payment-api | avg_duration=176.0
service=inventory | avg_duration=57.5


## Stream it (SSE)

The production UI consumes the same `progress → start → token → rows → done` events.

In [5]:
with client.stream("POST", "/ask/stream", json={
        "question": "How many errors per service?",
        "connection_id": "conn_demo"}) as resp:
    for line in resp.iter_lines():
        if line.startswith("data: "):
            print(line[6:][:110])

{"type": "progress", "stage": "connecting", "message": "Resolving connection and schema\u2026"}
{"type": "progress", "stage": "planning", "message": "Planning a validated query\u2026"}
{"type": "progress", "stage": "planning", "message": "Decomposing into sub-questions\u2026"}
{"type": "progress", "stage": "executing", "message": "Executing validated query\u2026"}
{"type": "start", "conversation_id": null, "columns": ["service", "error_count"], "row_count": 3}
{"type": "progress", "stage": "explaining", "message": "Generating answer\u2026"}
{"type": "token", "content": "Stub explanation."}
{"type": "rows", "rows": [{"service": "payment-api", "error_count": 1}, {"service": "order-service", "error_co
{"type": "verification", "ok": true, "note": "stub verification passed"}
{"type": "progress", "stage": "done", "message": "Complete"}
{"type": "done"}


## Inspect what persisted

Conversations, checkpoints, and audit records land in SQLite next to the demo.

In [6]:
print("checkpoints:", [(c["question"], c["row_count"]) for c in client.get("/checkpoints").json()])

checkpoints: [('How many errors per service?', 3), ('What is the average duration per service?', 4), ('How many traces are there?', 1), ('What is the average duration per service tier?', 3), ('How many errors per service?', 3), ('What is the average duration per service?', 3), ('How many traces are there?', 3), ('What is the average duration per service tier?', 3), ('How many errors per service?', 3), ('What is the average duration per service?', 3), ('How many traces are there?', 3), ('What is the average duration per service tier?', 3), ('How many errors per service?', 3), ('What is the average duration per service?', 3), ('How many traces are there?', 3), ('What is the average duration per service tier?', 3), ('How many errors per service?', 3), ('What is the average duration per service?', 3), ('How many traces are there?', 3)]
